# 🎭 Restaurant Atmosphere Classification
### LA Luxury Restaurant Recommendation System — Phase 4

**Purpose:** Replace emotion/sentiment scoring with a dedicated **restaurant
atmosphere classifier**, using zero-shot NLI — no fine-tuning required.

This notebook does two things in one run:
1. **Builds** `restaurants_with_atmosphere.csv` by running every restaurant through
   `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` against 13 custom atmosphere labels
2. **Validates** its own output — schema, nulls, label validity, confidence bounds,
   and metadata enrichment — before declaring the CSV ready for the Gradio dashboard

| Layer | Columns Added | Model | Input Text |
|-------|--------------|-------|------------|
| **Atmosphere Classification** | `predicted_atmosphere`, `atmosphere_confidence`, `secondary_atmosphere` | `MoritzLaurer/deberta-v3-base-zeroshot-v2.0` | `restaurant_metadata` (zero-shot, multi-label) |

**Why atmosphere instead of emotion:**  
Restaurant descriptions are factual and consistently positive in tone — emotion
classifiers (joy/anger/fear/etc.) add little signal because almost everything scores
as `neutral` or `joy`. **Atmosphere** is the dimension users actually search by —
*"romantic candlelit dinner,"* *"hip trendy bar scene,"* *"cozy intimate spot"* — and
it maps directly onto filters in the Gradio UI.

**Input:**  `restaurants_with_classifications.csv`  
**Output:** `restaurants_with_atmosphere.csv` — the final fully-enriched dataset for the Gradio UI

---
**Pipeline Overview:**
```
restaurants_with_classifications.csv  (21 columns)
        │
        ├── restaurant_metadata  ──► MoritzLaurer/deberta-v3-base-zeroshot-v2.0
        │   (zero-shot, multi_label=True)        │
        │                                         ▼
        │                          13 independent atmosphere scores
        │                          (Romantic, Fine Dining/Formal, Cozy/Intimate, ...)
        │                                         │
        │                          predicted_atmosphere (top label)
        │                          atmosphere_confidence (top score)
        │                          secondary_atmosphere (runner-up label)
        │
        ▼
  restaurants_with_atmosphere.csv  (24 columns — final enriched dataset)
        │
        ▼
  ✅ Self-validation (schema, nulls, labels, confidence bounds, metadata tags)
        │
        ▼
  🚀 Ready for GradioDashboard.py — Atmosphere dropdown filter
```

> **Note:** First run downloads `MoritzLaurer/deberta-v3-base-zeroshot-v2.0`
> (~370MB) from HuggingFace — requires internet access. Subsequent runs load
> from the local HuggingFace cache.

## 📦 Cell 1 — Import Libraries

In [1]:
import os
import time
import warnings
from unittest.mock import MagicMock

import pandas as pd
import numpy as np
from tqdm import tqdm

import torch
from transformers import pipeline

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 240)

# Auto-detect best available compute device
if torch.cuda.is_available():
    DEVICE = 0
    device_label = f"GPU — {torch.cuda.get_device_name(0)}"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    device_label = "Apple Silicon MPS"
else:
    DEVICE = "cpu"
    device_label = "CPU"

print(f"✅ Libraries loaded.")
print(f"   PyTorch version  : {torch.__version__}")
print(f"   Inference device : {device_label}")
print(f"   Note: CPU is fine — 94 restaurants processes in ~3-6 minutes.")

✅ Libraries loaded.
   PyTorch version  : 2.10.0+cpu
   Inference device : CPU
   Note: CPU is fine — 94 restaurants processes in ~3-6 minutes.


## 📂 Cell 2 — Load the Classified Restaurant Dataset

> **Note:** Place `restaurants_with_classifications.csv` in the same
> `data/` folder used by Notebooks 1–3.

In [2]:
INPUT_CSV  = "../data/restaurants_with_classifications.csv"
OUTPUT_CSV = "../data/restaurants_with_atmosphere.csv"

assert os.path.exists(INPUT_CSV), (
    f"Input CSV not found: {INPUT_CSV}\n"
    f"Run Notebook 3 (3_Restaurant_Text_Classification.ipynb) first."
)

restaurants = pd.read_csv(INPUT_CSV)

print(f"✅ Dataset loaded: {len(restaurants)} restaurants, {len(restaurants.columns)} columns")
print(f"\nExisting columns from Phase 3:")
for i, col in enumerate(restaurants.columns, 1):
    print(f"   {i:>2}. {col}")

✅ Dataset loaded: 94 restaurants, 21 columns

Existing columns from Phase 3:
    1. Name
    2. Location
    3. Description
    4. Address
    5. Telephone Number
    6. Price
    7. Cuisine Type
    8. Dining Atmosphere
    9. Sky-High Rooftop
   10. Michelin-Guide
   11. Customer Ratings
   12. Operation Hours
   13. Reservations
   14. Dress Code
   15. restaurant_metadata
   16. simple_cuisine_group
   17. dining_format
   18. predicted_occasion
   19. occasion_confidence
   20. predicted_vibe
   21. vibe_confidence


## 🏷️ Cell 3 — Define Atmosphere Labels

Sourced directly from `Restaurant_Atmosphere_List.docx`.  
**Family-Friendly** and **Rustic/Farmhouse** are excluded — no restaurant in the
71-row dataset matches either profile, and including them only adds noise to
every classification score.

In [3]:
ATMOSPHERE_LABELS = [
    "Romantic",                   # Dim lighting, candles, intimate seating
    "Energetic / Lively",         # Bustling, louder music, high energy
    "Casual",                     # Relaxed, unpretentious, minimal formality
    "Fine Casual",                # Elevated quality, laid-back formality
    "Fine Dining / Formal",       # Elegant decor, hushed, white-glove service
    "Cozy / Intimate",            # Warm, small spaces, homey feel
    "Trendy / Hip",               # Modern design, Instagram-worthy, curated music
    "Industrial / Urban",         # Exposed brick, metal, warehouse-style
    "Minimalist / Modern",        # Clean lines, neutral palette, uncluttered
    "Traditional / Classic",      # Timeless decor, historical/cultural elements
    "Theatrical / Entertainment", # Dining as performance, interactive elements
    "Beachy / Tropical",          # Light, airy, nautical or island-inspired
    "Upscale Casual",             # Polished but approachable, not stuffy
]

print(f"✅ {len(ATMOSPHERE_LABELS)} atmosphere labels loaded:")
for label in ATMOSPHERE_LABELS:
    print(f"   • {label}")

✅ 13 atmosphere labels loaded:
   • Romantic
   • Energetic / Lively
   • Casual
   • Fine Casual
   • Fine Dining / Formal
   • Cozy / Intimate
   • Trendy / Hip
   • Industrial / Urban
   • Minimalist / Modern
   • Traditional / Classic
   • Theatrical / Entertainment
   • Beachy / Tropical
   • Upscale Casual


## ⚙️ Cell 4 — Load the Zero-Shot Atmosphere Classifier

`MoritzLaurer/deberta-v3-base-zeroshot-v2.0` is the current best zero-shot
classification model for custom English label sets:
- Outperforms `facebook/bart-large-mnli` on custom label sets
- ~86M parameters (DeBERTa-v3-base) — faster than bart-large-mnli
- MIT license — fully commercial-friendly
- Requires **zero fine-tuning** and **zero labeled training data**

In [4]:
ATMOSPHERE_MODEL = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"

print(f"⏳ Loading atmosphere model: {ATMOSPHERE_MODEL}")
print("   First run downloads ~370MB — subsequent runs load from HuggingFace cache.")
print()

atmosphere_classifier = pipeline(
    "zero-shot-classification",
    model=ATMOSPHERE_MODEL,
    device=DEVICE,
)

print(f"✅ Atmosphere model loaded on device: {DEVICE}")

# Smoke test on a restaurant-flavoured sentence
test_sentence = (
    "A candlelit restaurant with dim lighting and intimate seating for two, "
    "quiet ambiance throughout the dining room."
)
test_result = atmosphere_classifier(
    test_sentence,
    candidate_labels=ATMOSPHERE_LABELS,
    multi_label=True,
)

print(f"\nSmoke test: '{test_sentence[:60]}...'")
print(f"  Top atmosphere: {test_result['labels'][0]} ({test_result['scores'][0]:.4f}) ✅")
print(f"  Top 5 scores:")
for label, score in zip(test_result['labels'][:5], test_result['scores'][:5]):
    bar = '█' * int(score * 20)
    print(f"    {label:<28} {bar} {score:.4f}")

⏳ Loading atmosphere model: MoritzLaurer/deberta-v3-base-zeroshot-v2.0
   First run downloads ~370MB — subsequent runs load from HuggingFace cache.



config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

✅ Atmosphere model loaded on device: cpu

Smoke test: 'A candlelit restaurant with dim lighting and intimate seatin...'
  Top atmosphere: Cozy / Intimate (0.9959) ✅
  Top 5 scores:
    Cozy / Intimate              ███████████████████ 0.9959
    Romantic                     ███████████████████ 0.9957
    Fine Dining / Formal         ███████████████████ 0.9689
    Traditional / Classic        ███████████████████ 0.9578
    Casual                       █████████████████ 0.8531


## 🛠️ Cell 5 — Define the Classification Wrapper

`classify_atmosphere()` takes a restaurant's `restaurant_metadata` string and
returns three values:
- `predicted_atmosphere` — highest-scoring label
- `atmosphere_confidence` — confidence score for the top label (rounded to 4dp)
- `secondary_atmosphere` — runner-up label (second-highest score)

We use `multi_label=True` because a restaurant can authentically be both
**Romantic** and **Cozy/Intimate**, or both **Trendy/Hip** and **Industrial/Urban**.
Scores are independent probabilities, not a forced single-class softmax.

In [5]:
def classify_atmosphere(classifier, text: str, candidate_labels: list) -> tuple:
    """
    Zero-shot atmosphere classification for a single restaurant.

    Returns
    -------
    predicted_atmosphere  : str   — top-scoring label
    atmosphere_confidence : float — confidence score, rounded to 4 decimal places
    secondary_atmosphere  : str   — runner-up label (falls back to primary if
                                     only one label is returned)
    """
    result = classifier(
        text,
        candidate_labels=candidate_labels,
        multi_label=True,
    )
    predicted_atmosphere  = result["labels"][0]
    atmosphere_confidence = round(result["scores"][0], 4)
    secondary_atmosphere  = (
        result["labels"][1] if len(result["labels"]) > 1 else predicted_atmosphere
    )
    return predicted_atmosphere, atmosphere_confidence, secondary_atmosphere


# ── Sanity check against a real restaurant in the dataset ─────────────────────
sample_row = restaurants.iloc[5]   # Vespertine
atm, conf, sec = classify_atmosphere(
    atmosphere_classifier, sample_row["restaurant_metadata"], ATMOSPHERE_LABELS
)
print(f"{sample_row['Name']} → Primary: '{atm}' ({conf:.4f}) | Secondary: '{sec}'")
print(f"\n✅ Wrapper function ready.")

Vespertine → Primary: 'Trendy / Hip' (0.9503) | Secondary: 'Fine Dining / Formal'

✅ Wrapper function ready.


## 🔄 Cell 6 — Dry Run: Atmosphere Classification on First 5 Restaurants

Validates the pipeline on a small sample before running all 71.

In [6]:
print("=== DRY RUN: First 5 restaurants ===")
print(f"{'Restaurant':<22} {'Primary Atmosphere':<26} {'Conf':>6}  {'Secondary'}")
print("-" * 80)

for i in range(5):
    row = restaurants.iloc[i]
    atm, conf, sec = classify_atmosphere(
        atmosphere_classifier, row["restaurant_metadata"], ATMOSPHERE_LABELS
    )
    print(f"  {row['Name']:<20} {atm:<26} {conf:>6.4f}  {sec}")

print("\n✅ Dry run successful. Running full dataset in Cell 7.")

=== DRY RUN: First 5 restaurants ===
Restaurant             Primary Atmosphere           Conf  Secondary
--------------------------------------------------------------------------------
  Somni                Fine Dining / Formal       0.9989  Upscale Casual
  Providence           Fine Dining / Formal       0.9792  Trendy / Hip
  Hayato               Fine Dining / Formal       0.9866  Upscale Casual
  n/naka               Fine Dining / Formal       0.9952  Upscale Casual
  Melisse              Fine Dining / Formal       0.9992  Upscale Casual

✅ Dry run successful. Running full dataset in Cell 7.


## 🔄 Cell 7 — Run Atmosphere Classification on All 71 Restaurants

Expected runtime: **3–6 minutes** on CPU.

In [7]:
predicted_atmospheres  = []
atmosphere_confidences = []
secondary_atmospheres  = []

print(f"⏳ Running atmosphere classification on all {len(restaurants)} restaurants...")
print(f"   Model : {ATMOSPHERE_MODEL}")
print(f"   Input : restaurant_metadata (multi-label zero-shot)\n")

start_time = time.time()

for idx in tqdm(range(len(restaurants)), desc="Atmosphere Classification"):
    row = restaurants.iloc[idx]
    atm, conf, sec = classify_atmosphere(
        atmosphere_classifier, row["restaurant_metadata"], ATMOSPHERE_LABELS
    )
    predicted_atmospheres.append(atm)
    atmosphere_confidences.append(conf)
    secondary_atmospheres.append(sec)

elapsed = time.time() - start_time
print(f"\n✅ Atmosphere classification complete for all {len(restaurants)} restaurants.")
print(f"   Elapsed: {elapsed:.1f}s")

⏳ Running atmosphere classification on all 94 restaurants...
   Model : MoritzLaurer/deberta-v3-base-zeroshot-v2.0
   Input : restaurant_metadata (multi-label zero-shot)



Atmosphere Classification: 100%|██████████| 94/94 [25:48<00:00, 16.48s/it]


✅ Atmosphere classification complete for all 94 restaurants.
   Elapsed: 1548.8s


## 📥 Cell 8 — Attach Results to DataFrame

In [8]:
restaurants["predicted_atmosphere"]  = predicted_atmospheres
restaurants["atmosphere_confidence"] = atmosphere_confidences
restaurants["secondary_atmosphere"]  = secondary_atmospheres

print(f"✅ New columns added. Shape is now: {restaurants.shape}")
print(f"\nSample (first 10 restaurants):")
print(restaurants[["Name", "predicted_atmosphere", "atmosphere_confidence", "secondary_atmosphere"]]
      .head(10).to_string())

✅ New columns added. Shape is now: (94, 24)

Sample (first 10 restaurants):
              Name  predicted_atmosphere  atmosphere_confidence  secondary_atmosphere
0            Somni  Fine Dining / Formal                 0.9989        Upscale Casual
1       Providence  Fine Dining / Formal                 0.9792          Trendy / Hip
2           Hayato  Fine Dining / Formal                 0.9866        Upscale Casual
3           n/naka  Fine Dining / Formal                 0.9952        Upscale Casual
4          Melisse  Fine Dining / Formal                 0.9992        Upscale Casual
5       Vespertine          Trendy / Hip                 0.9503  Fine Dining / Formal
6  Sushi Kaneyoshi  Fine Dining / Formal                 0.9849       Cozy / Intimate
7    Restaurant Ki  Fine Dining / Formal                 0.9526       Cozy / Intimate
8        715 Sushi  Fine Dining / Formal                 0.9904   Minimalist / Modern
9   Orsa & Winston  Fine Dining / Formal                 0.9973 

## 📊 Cell 9 — Distribution Analysis

In [9]:
print("=" * 55)
print("PRIMARY ATMOSPHERE DISTRIBUTION")
print("=" * 55)
atm_counts = restaurants["predicted_atmosphere"].value_counts()
for label, count in atm_counts.items():
    bar = "█" * count
    print(f"  {label:<28} {count:2d}  {bar}")

print(f"\nTotal restaurants classified : {len(restaurants)}")
print(f"Unique atmosphere labels used: {restaurants['predicted_atmosphere'].nunique()}")

print("\n" + "=" * 55)
print("CONFIDENCE SCORE SUMMARY")
print("=" * 55)
print(restaurants["atmosphere_confidence"].describe().round(4).to_string())

print("\n" + "=" * 55)
print("SECONDARY ATMOSPHERE DISTRIBUTION")
print("=" * 55)
sec_counts = restaurants["secondary_atmosphere"].value_counts()
for label, count in sec_counts.items():
    print(f"  {label:<28} {count:2d}")

PRIMARY ATMOSPHERE DISTRIBUTION
  Fine Dining / Formal         53  █████████████████████████████████████████████████████
  Casual                       15  ███████████████
  Trendy / Hip                 14  ██████████████
  Romantic                      6  ██████
  Fine Casual                   2  ██
  Minimalist / Modern           2  ██
  Upscale Casual                2  ██

Total restaurants classified : 94
Unique atmosphere labels used: 7

CONFIDENCE SCORE SUMMARY
count    94.0000
mean      0.9534
std       0.0768
min       0.5024
25%       0.9504
50%       0.9802
75%       0.9948
max       0.9994

SECONDARY ATMOSPHERE DISTRIBUTION
  Upscale Casual               41
  Trendy / Hip                  9
  Traditional / Classic         9
  Cozy / Intimate               8
  Minimalist / Modern           7
  Industrial / Urban            6
  Romantic                      4
  Fine Casual                   4
  Fine Dining / Formal          2
  Energetic / Lively            2
  Casual         

## 🏷️ Cell 10 — Enrich `restaurant_metadata` with Atmosphere Tags

Appends three tags to each restaurant's `restaurant_metadata` string so that
ChromaDB semantic search can surface atmosphere when users query phrases like
*"romantic candlelit dinner"* or *"hip trendy bar scene."*

In [10]:
def append_atmosphere_tags(row: pd.Series) -> str:
    """Appends Primary/Secondary Atmosphere + Confidence tags to restaurant_metadata."""
    tags = (
        f" Primary Atmosphere: {row['predicted_atmosphere']}."
        f" Secondary Atmosphere: {row['secondary_atmosphere']}."
        f" Atmosphere Confidence: {row['atmosphere_confidence']}."
    )
    return row["restaurant_metadata"] + tags


restaurants["restaurant_metadata"] = restaurants.apply(append_atmosphere_tags, axis=1)

# Spot-check
sample = restaurants[restaurants["Name"] == "Vespertine"]["restaurant_metadata"].iloc[0]
print("Vespertine metadata tail (last 200 chars):")
print("  ..." + sample[-200:])

print(f"\n✅ restaurant_metadata enriched for all {len(restaurants)} restaurants")

Vespertine metadata tail (last 200 chars):
  ...rary American. Dining Format: Full Service. Best For: Special Occasion. Vibe: Hip & Trendy. Primary Atmosphere: Trendy / Hip. Secondary Atmosphere: Fine Dining / Formal. Atmosphere Confidence: 0.9503.

✅ restaurant_metadata enriched for all 94 restaurants


## 💾 Cell 11 — Save `restaurants_with_atmosphere.csv`

This is the file the Gradio dashboard will load instead of
`restaurants_with_emotions.csv`.

In [11]:
OUTPUT_PATH = "../data/restaurants_with_atmosphere.csv"
restaurants.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Saved: {OUTPUT_PATH}")
print(f"   Rows    : {len(restaurants)}")
print(f"   Columns : {len(restaurants.columns)}")
print(f"\nFinal column list:")
for i, col in enumerate(restaurants.columns, 1):
    tag = "[Phase 4 ← NEW]" if col in (
        "predicted_atmosphere", "atmosphere_confidence", "secondary_atmosphere"
    ) else ""
    print(f"   {i:>2}. {col:<25} {tag}")

✅ Saved: ../data/restaurants_with_atmosphere.csv
   Rows    : 94
   Columns : 24

Final column list:
    1. Name                      
    2. Location                  
    3. Description               
    4. Address                   
    5. Telephone Number          
    6. Price                     
    7. Cuisine Type              
    8. Dining Atmosphere         
    9. Sky-High Rooftop          
   10. Michelin-Guide            
   11. Customer Ratings          
   12. Operation Hours           
   13. Reservations              
   14. Dress Code                
   15. restaurant_metadata       
   16. simple_cuisine_group      
   17. dining_format             
   18. predicted_occasion        
   19. occasion_confidence       
   20. predicted_vibe            
   21. vibe_confidence           
   22. predicted_atmosphere      [Phase 4 ← NEW]
   23. atmosphere_confidence     [Phase 4 ← NEW]
   24. secondary_atmosphere      [Phase 4 ← NEW]


---
## ✅ SELF-VALIDATION — Confirming the CSV is Ready for Gradio

The cells below re-load `restaurants_with_atmosphere.csv` from disk
(the file just saved above) and run the same checks as the project's
test suite — schema, nulls, label validity, confidence bounds, and
metadata enrichment — so you know the file is production-ready before
wiring it into `GradioDashboard.py`.

**No mocks here — these are real assertions against the real output.**

## 📐 Cell 12 — Reload CSV & Validate Schema

In [16]:
# Reload from disk to confirm the saved file is actually correct,
# not just the in-memory DataFrame we built it from.
df = pd.read_csv(OUTPUT_PATH)

EXPECTED_COLUMNS = [
    "Name", "Location", "Description", "Address", "Telephone Number",
    "Price", "Cuisine Type", "Dining Atmosphere", "Sky-High Rooftop",
    "Michelin-Guide", "Customer Ratings", "Operation Hours",
    "Reservations", "Dress Code", "restaurant_metadata",
    "simple_cuisine_group", "dining_format", "predicted_occasion",
    "occasion_confidence", "predicted_vibe", "vibe_confidence",
    "predicted_atmosphere", "atmosphere_confidence", "secondary_atmosphere",
]

print("=" * 60)
print("SCHEMA VALIDATION")
print("=" * 60)

schema_errors = []

if len(df.columns) == 24:
    print(f"  ✅ Column count: {len(df.columns)}  (21 prior + 3 new)")
else:
    schema_errors.append(f"Column count: expected 24, got {len(df.columns)}")
    print(f"  ❌ Column count: expected 24, got {len(df.columns)}")

if list(df.columns) == EXPECTED_COLUMNS:
    print(f"  ✅ Column order matches expected exactly")
else:
    schema_errors.append("Column order mismatch")
    print(f"  ❌ Column order mismatch")

if len(df) == 94:
    print(f"  ✅ Row count: {len(df)}")
else:
    schema_errors.append(f"Row count: expected 94, got {len(df)}")
    print(f"  ❌ Row count: expected 94, got {len(df)}")

print()
if not schema_errors:
    print("  ✅ All schema checks passed — saved CSV matches expected structure.")
else:
    print(f"  ❌ {len(schema_errors)} schema issue(s) found:")
    for e in schema_errors:
        print(f"     • {e}")

SCHEMA VALIDATION
  ✅ Column count: 24  (21 prior + 3 new)
  ✅ Column order matches expected exactly
  ✅ Row count: 94

  ✅ All schema checks passed — saved CSV matches expected structure.


## 🚫 Cell 13 — Null Checks & Label Validity

In [17]:
print("=" * 60)
print("NULL CHECKS & LABEL VALIDITY")
print("=" * 60)

value_errors = []

for col in ["predicted_atmosphere", "atmosphere_confidence", "secondary_atmosphere"]:
    nulls = df[col].isnull().sum()
    if nulls == 0:
        print(f"  ✅ {col:<25} — 0 nulls")
    else:
        value_errors.append(f"{col}: {nulls} nulls")
        print(f"  ❌ {col}: {nulls} nulls")

unknown_labels = set(df["predicted_atmosphere"].unique()) - set(ATMOSPHERE_LABELS)
if not unknown_labels:
    print(f"  ✅ predicted_atmosphere — all values are known labels")
else:
    value_errors.append(f"Unknown labels: {unknown_labels}")
    print(f"  ❌ Unknown predicted_atmosphere labels: {unknown_labels}")

oob = df[(df["atmosphere_confidence"] < 0) | (df["atmosphere_confidence"] > 1)]
if oob.empty:
    cmin, cmax = df["atmosphere_confidence"].min(), df["atmosphere_confidence"].max()
    print(f"  ✅ atmosphere_confidence — all in [0,1]  (range {cmin:.4f}–{cmax:.4f})")
else:
    value_errors.append(f"{len(oob)} confidence values out of [0,1]")
    print(f"  ❌ {len(oob)} confidence values out of [0,1]")

print()
if not value_errors:
    print("  ✅ All null & label validity checks passed.")
else:
    print(f"  ❌ {len(value_errors)} issue(s) found.")

NULL CHECKS & LABEL VALIDITY
  ✅ predicted_atmosphere      — 0 nulls
  ✅ atmosphere_confidence     — 0 nulls
  ✅ secondary_atmosphere      — 0 nulls
  ✅ predicted_atmosphere — all values are known labels
  ✅ atmosphere_confidence — all in [0,1]  (range 0.5024–0.9994)

  ✅ All null & label validity checks passed.


## 🏷️ Cell 14 — Metadata Enrichment Validation

In [ ]:
print("=" * 60)
print("METADATA ENRICHMENT VALIDATION")
print("=" * 60)

meta_errors = []

missing_primary = [
    row["Name"] for _, row in df.iterrows()
    if f"Primary Atmosphere: {row['predicted_atmosphere']}" not in row["restaurant_metadata"]
]
if not missing_primary:
    print("  ✅ Primary Atmosphere tag present in all 94 metadata strings")
else:
    meta_errors.append(f"Missing Primary Atmosphere tag: {missing_primary}")
    print(f"  ❌ Missing Primary Atmosphere tag for: {missing_primary}")

missing_secondary = [
    row["Name"] for _, row in df.iterrows()
    if f"Secondary Atmosphere: {row['secondary_atmosphere']}" not in row["restaurant_metadata"]
]
if not missing_secondary:
    print("  ✅ Secondary Atmosphere tag present in all 94 metadata strings")
else:
    meta_errors.append(f"Missing Secondary Atmosphere tag: {missing_secondary}")
    print(f"  ❌ Missing Secondary Atmosphere tag for: {missing_secondary}")

print()
if not meta_errors:
    print("  ✅ Metadata enrichment confirmed for all restaurants.")
else:
    print(f"  ❌ {len(meta_errors)} issue(s) found.")

METADATA ENRICHMENT VALIDATION


NameError: name 'df' is not defined

## 📊 Cell 15 — Final Pipeline Summary & Gradio Integration Notes

In [19]:
print("=" * 66)
print("   ATMOSPHERE CLASSIFICATION PIPELINE — FINAL SUMMARY")
print("=" * 66)
print(f"  Total Restaurants          : {len(df)}")
print(f"  New Columns Added (Phase 4): 3")
print(f"  Total Columns in Output    : {len(df.columns)}")
print()
print("  Primary Atmosphere Distribution:")
for label, count in df["predicted_atmosphere"].value_counts().items():
    bar = "█" * count
    print(f"    {label:<28} {count:2d}  {bar}")
print()
print(f"  Confidence range: [{df['atmosphere_confidence'].min():.4f}, {df['atmosphere_confidence'].max():.4f}]")
print(f"  Confidence mean : {df['atmosphere_confidence'].mean():.4f}")
print()
print(f"  Output: {OUTPUT_PATH}")
print("=" * 66)
print("  ✅ Full atmosphere enrichment + self-validation complete!")
print("=" * 66)
print()
print("🚀 NEXT STEP: Wire into GradioDashboard.py")
print("   1. Change CSV_PATH to load 'restaurants_with_atmosphere.csv'")
print("      instead of 'restaurants_with_emotions.csv'")
print("   2. Add a new Atmosphere dropdown filter sourced from:")
print("        sorted(df['predicted_atmosphere'].unique())")
print("   3. Optionally surface 'secondary_atmosphere' on restaurant detail cards")
print("      (e.g. 'Romantic, with Cozy/Intimate undertones')")
print("   4. Combine with existing Phase 3 filters:")
print("        Cuisine Group  (simple_cuisine_group)")
print("        Dining Format  (dining_format)")
print("        Occasion       (predicted_occasion)")
print("        Vibe           (predicted_vibe)")

   ATMOSPHERE CLASSIFICATION PIPELINE — FINAL SUMMARY
  Total Restaurants          : 94
  New Columns Added (Phase 4): 3
  Total Columns in Output    : 24

  Primary Atmosphere Distribution:
    Fine Dining / Formal         53  █████████████████████████████████████████████████████
    Casual                       15  ███████████████
    Trendy / Hip                 14  ██████████████
    Romantic                      6  ██████
    Fine Casual                   2  ██
    Minimalist / Modern           2  ██
    Upscale Casual                2  ██

  Confidence range: [0.5024, 0.9994]
  Confidence mean : 0.9534

  Output: ../data/restaurants_with_atmosphere.csv
  ✅ Full atmosphere enrichment + self-validation complete!

🚀 NEXT STEP: Wire into GradioDashboard.py
   1. Change CSV_PATH to load 'restaurants_with_atmosphere.csv'
      instead of 'restaurants_with_emotions.csv'
   2. Add a new Atmosphere dropdown filter sourced from:
        sorted(df['predicted_atmosphere'].unique())
   3. Opt